# GtoPdb composite-target mapping EDA

This notebook evaluates the **consequences** of representing GtoPdb composite
targets, and provides an auditable entry point for evaluating an external
complex mapping.

It deliberately separates:

- **identity** — GtoPdb Target ID, external complex CURIE, and component IDs;
- **semantics** — a GtoPdb pharmacological assertion is about a source target,
  not automatically about each component;
- **evidence** — source target fields, species, and candidate external mapping;
- **policy** — exclusion, source-defined target nodes, validated cross-reference,
  external substitution, or component expansion.

## Decisions this notebook can support

| Policy | Pharmacology assertion attaches to | Principal benefit | Principal risk |
| --- | --- | --- | --- |
| Current exclusion | nothing for composites | no malformed CURIE or inflation | known source assertions are absent |
| Source-defined target | GtoPdb target node | preserves source scope | requires source-target node policy |
| External cross-reference | GtoPdb target, linked to external CURIE | enrichment without identity loss | mapping must be auditable |
| External substitution | external complex CURIE | interoperability | conflation / one-to-many mismatch |
| Component expansion | each component protein | apparent edge recovery | assertion inflation and semantic overclaim |

The preferred first implementation is **source-defined target identity plus a
validated external cross-reference**. Do not use component expansion unless the
source explicitly asserts component-level pharmacology.


In [1]:
from pathlib import Path
import pandas as pd

# Set this to a downloaded GtoPdb release directory. The current development
# analysis used /tmp/gtopdb-analysis with GtoPdb 2026.2 CSVs.
DATA_DIR = Path('/tmp/gtopdb-analysis')
INTERACTIONS = DATA_DIR / 'interactions.csv'
TARGETS_AND_FAMILIES = DATA_DIR / 'targets_and_families.csv'

# Optional, analyst-supplied external mapping. Required columns:
#   gtopdb_target_id, external_curie
# Recommended columns:
#   species, mapping_relation, evidence, confidence
EXTERNAL_COMPLEX_MAP = DATA_DIR / 'gtopdb_external_complex_map.csv'

assert INTERACTIONS.exists(), f'Missing {INTERACTIONS}'
interactions = pd.read_csv(INTERACTIONS, skiprows=1, low_memory=False)
interactions.columns.tolist(), interactions.shape


(['Target',
  'Target ID',
  'Target Subunit IDs',
  'Target Gene Symbol',
  'Target UniProt ID',
  'Target Ensembl Gene ID',
  'Target Ligand',
  'Target Ligand ID',
  'Target Ligand Subunit IDs',
  'Target Ligand Gene Symbol',
  'Target Ligand UniProt ID',
  'Target Ligand Ensembl Gene ID',
  'Target Ligand PubChem SID',
  'Target Species',
  'Ligand ID',
  'Ligand',
  'Ligand Type',
  'Ligand Subunit IDs',
  'Ligand Gene Symbol',
  'Ligand Species',
  'Ligand PubChem SID',
  'Approved',
  'Type',
  'Action',
  'Action comment',
  'Selectivity',
  'Endogenous',
  'Primary Target',
  'concentration Range',
  'Affinity Units',
  'Affinity High',
  'Affinity Median',
  'Affinity Low',
  'Original Affinity Units',
  'Original Affinity Low nm',
  'Original Affinity Median nm',
  'Original Affinity High nm',
  'Original Affinity Relation',
  'Assay Description',
  'Receptor Site',
  'Ligand Context',
  'PubMed ID',
  'Webpage URLs',
  'Patent Numbers'],
 (24599, 44))

## Identify invalid compound-UniProt fields and broader composite evidence

These are related but distinct scopes:

1. **Compound-UniProt defect** — `Target UniProt ID` contains multiple
   accessions. These values previously risked becoming an invalid CURIE such as
   `UniProtKB:P46098|O95264`.
2. **Broader composite evidence** — any of UniProt IDs, GtoPdb subunit IDs, or
   gene symbols contains multiple values. This broader boundary identifies
   source targets that should not be projected automatically as one protein,
   even when the UniProt field itself is missing or singular.

A pipe is evidence of multiple values, not sufficient evidence that the target
is specifically a physical complex.


In [2]:
TARGET_ID = 'Target ID'
TARGET_NAME = 'Target'
SPECIES = 'Target Species'
UNIPROT = 'Target UniProt ID'
SUBUNITS = 'Target Subunit IDs'
GENES = 'Target Gene Symbol'


def pipe_values(value):
    if pd.isna(value):
        return ()
    return tuple(part.strip() for part in str(value).split('|') if part.strip())


uniprot_values = interactions[UNIPROT].map(pipe_values)
subunit_values = interactions[SUBUNITS].map(pipe_values)
gene_values = interactions[GENES].map(pipe_values)

compound_uniprot_mask = uniprot_values.map(len) > 1
broader_composite_mask = (
    compound_uniprot_mask
    | (subunit_values.map(len) > 1)
    | (gene_values.map(len) > 1)
)

compound_uniprot_rows = interactions.loc[compound_uniprot_mask].copy()
compound_uniprot_rows['uniprot_components'] = uniprot_values[compound_uniprot_mask]
composite_rows = interactions.loc[broader_composite_mask].copy()
composite_rows['uniprot_components'] = uniprot_values[broader_composite_mask]

compound_uniprot_relationships = {
    (
        str(row[TARGET_ID]),
        row[SPECIES] if pd.notna(row[SPECIES]) else None,
        component,
    )
    for _, row in compound_uniprot_rows.iterrows()
    for component in pipe_values(row[UNIPROT])
}

summary = pd.Series({
    'all interaction rows': len(interactions),
    'compound-UniProt rows': len(compound_uniprot_rows),
    'compound-UniProt target concepts': compound_uniprot_rows[TARGET_ID].nunique(),
    'compound-UniProt target/species realizations': compound_uniprot_rows[[TARGET_ID, SPECIES]].drop_duplicates().shape[0],
    'compound-UniProt component relationships': len(compound_uniprot_relationships),
    'broader composite-evidence rows': len(composite_rows),
    'broader composite-evidence target concepts': composite_rows[TARGET_ID].nunique(),
    'broader target/species realizations': composite_rows[[TARGET_ID, SPECIES]].drop_duplicates().shape[0],
    'additional broader-evidence rows': len(composite_rows) - len(compound_uniprot_rows),
})
summary


all interaction rows                            24599
compound-UniProt rows                             379
compound-UniProt target concepts                   66
compound-UniProt target/species realizations       79
compound-UniProt component relationships          190
broader composite-evidence rows                   386
broader composite-evidence target concepts         67
broader target/species realizations                85
additional broader-evidence rows                    7
dtype: int64

## Current policy: exclude composite targets

The production boundary excludes these records because converting a component
list such as `P46098|O95264` into `UniProtKB:P46098|O95264` emits an invalid
CURIE. This policy is intentionally conservative: it loses target-level
pharmacology assertions rather than inventing component-level ones.


In [3]:
# The rows currently excluded before graph construction.
composite_rows[[TARGET_ID, TARGET_NAME, SPECIES, UNIPROT, SUBUNITS, GENES]].drop_duplicates().head(10)


,Target ID,Target,Target Species,Target UniProt ID,Target Subunit IDs,Target Gene Symbol
921,378.0,5-HT<sub>3</sub>AB,Human,P46098|O95264,373|374,HTR3A|HTR3B
930,378.0,5-HT<sub>3</sub>AB,Mouse,P23979|Q9JHJ5,373|374,Htr3a|Htr3b
1475,2588.0,Activin receptors,Human,P27037|P36896|Q13705|Q8NER5,1787|1790|1791|1792,ACVR1B|ACVR1C|ACVR2A|ACVR2B
2153,49.0,AM<sub>1</sub> receptor,Human,Q16602|O60895,47|52,CALCRL|RAMP2
2154,49.0,AM<sub>1</sub> receptor,Rat,Q63118|Q9JHJ1,47|52,Calcrl|Ramp2
2155,49.0,AM<sub>1</sub> receptor,Mouse,Q9R1W5|Q9WUP0,47|52,Calcrl|Ramp2
2168,49.0,AM<sub>1</sub> receptor,NaN,NaN,47|52,NaN
2169,50.0,AM<sub>2</sub> receptor,Mouse,Q9R1W5|Q9WUP1,47|53,Calcrl|Ramp3
2170,50.0,AM<sub>2</sub> receptor,Rat,Q63118|Q9JJ73,47|53,Calcrl|Ramp3
2171,50.0,AM<sub>2</sub> receptor,Human,Q16602|O60896,47|53,CALCRL|RAMP3


## Scenario accounting

The table distinguishes the full conservative target boundary from the narrower
compound-UniProt defect. Counts are raw source-row upper bounds; normalization
and aggregation determine final graph-edge recovery.

Component expansion applies only to the 379 rows with multiple UniProt
accessions. The seven additional broader-evidence rows cannot be repaired by
splitting a compound UniProt value because they do not have one.


In [4]:
compound_expansion_rows = compound_uniprot_rows['uniprot_components'].map(len).sum()
additional_component_assertions = compound_expansion_rows - len(compound_uniprot_rows)

scenario = pd.DataFrame(
    [
        {
            'policy': 'Current conservative exclusion',
            'source composite rows addressed': len(composite_rows),
            'target-level pharmacology rows retained': 0,
            'component-level pharmacology rows emitted': 0,
            'unresolved broader-evidence rows': len(composite_rows),
            'interpretation': 'No malformed CURIE or inflation; assertions absent.',
        },
        {
            'policy': 'Source-defined GtoPdb target',
            'source composite rows addressed': len(composite_rows),
            'target-level pharmacology rows retained': len(composite_rows),
            'component-level pharmacology rows emitted': 0,
            'unresolved broader-evidence rows': 0,
            'interpretation': 'Preserves source assertion scope.',
        },
        {
            'policy': 'Validated external complex cross-reference',
            'source composite rows addressed': len(composite_rows),
            'target-level pharmacology rows retained': len(composite_rows),
            'component-level pharmacology rows emitted': 0,
            'unresolved broader-evidence rows': 0,
            'interpretation': 'Same assertions; adds validated interoperability.',
        },
        {
            'policy': 'Unsafe compound-UniProt expansion',
            'source composite rows addressed': len(compound_uniprot_rows),
            'target-level pharmacology rows retained': 0,
            'component-level pharmacology rows emitted': compound_expansion_rows,
            'unresolved broader-evidence rows': len(composite_rows) - len(compound_uniprot_rows),
            'interpretation': f'Creates {additional_component_assertions} extra component assertions and leaves broader cases unresolved.',
        },
    ]
)
scenario


,policy,source composite rows addressed,target-level pharmacology rows retained,component-level pharmacology rows emitted,unresolved broader-evidence rows,interpretation
0,Current conservative exclusion,386,0,0,386,No malformed CURIE or inflation; assertions ab...
1,Source-defined GtoPdb target,386,386,0,0,Preserves source assertion scope.
2,Validated external complex cross-reference,386,386,0,0,Same assertions; adds validated interoperability.
3,Unsafe compound-UniProt expansion,379,0,968,7,Creates 589 extra component assertions and lea...


## External complex-map evaluation

Create an optional CSV at `EXTERNAL_COMPLEX_MAP` with one row per proposed
mapping. Do not accept a match merely because component strings overlap.

A candidate should be reviewed as:

- **exact-equivalent** — same source target, taxon, and component semantics;
- **cross-reference** — useful external identifier but GtoPdb identity remains
  primary;
- **ambiguous** — one-to-many/many-to-one, species mismatch, or partial
  component overlap; retain unresolved;
- **unmapped** — retain GtoPdb identity without external enrichment.


In [5]:
target_inventory = (
    composite_rows[[TARGET_ID, TARGET_NAME, SPECIES, UNIPROT, SUBUNITS, GENES]]
    .drop_duplicates()
    .assign(component_count=lambda frame: frame[UNIPROT].map(pipe_values).map(len))
)

if EXTERNAL_COMPLEX_MAP.exists():
    external_map = pd.read_csv(EXTERNAL_COMPLEX_MAP, dtype={TARGET_ID: 'string', 'external_curie': 'string'})
    required = {TARGET_ID, 'external_curie'}
    missing = required - set(external_map.columns)
    assert not missing, f'External map is missing required columns: {missing}'

    mapping_cardinality = external_map.groupby(TARGET_ID)['external_curie'].nunique()
    mapping_report = target_inventory.merge(external_map, on=TARGET_ID, how='left')
    coverage = pd.Series({
        'composite GtoPdb targets': len(target_inventory),
        'targets with an external candidate': mapping_cardinality.size,
        'one-to-one candidates': (mapping_cardinality == 1).sum(),
        'one-to-many candidates requiring review': (mapping_cardinality > 1).sum(),
        'unmapped targets': len(target_inventory) - mapping_cardinality.size,
    })
    display(coverage)
    display(mapping_report.sort_values([TARGET_ID, 'external_curie']).head(20))
else:
    print(f'No external map found at {EXTERNAL_COMPLEX_MAP}.')
    print('Export target_inventory, curate candidate mappings, then rerun this cell.')

target_inventory.head(10)


No external map found at /tmp/gtopdb-analysis/gtopdb_external_complex_map.csv.
Export target_inventory, curate candidate mappings, then rerun this cell.


,Target ID,Target,Target Species,Target UniProt ID,Target Subunit IDs,Target Gene Symbol,component_count
921,378.0,5-HT<sub>3</sub>AB,Human,P46098|O95264,373|374,HTR3A|HTR3B,2
930,378.0,5-HT<sub>3</sub>AB,Mouse,P23979|Q9JHJ5,373|374,Htr3a|Htr3b,2
1475,2588.0,Activin receptors,Human,P27037|P36896|Q13705|Q8NER5,1787|1790|1791|1792,ACVR1B|ACVR1C|ACVR2A|ACVR2B,4
2153,49.0,AM<sub>1</sub> receptor,Human,Q16602|O60895,47|52,CALCRL|RAMP2,2
2154,49.0,AM<sub>1</sub> receptor,Rat,Q63118|Q9JHJ1,47|52,Calcrl|Ramp2,2
2155,49.0,AM<sub>1</sub> receptor,Mouse,Q9R1W5|Q9WUP0,47|52,Calcrl|Ramp2,2
2168,49.0,AM<sub>1</sub> receptor,NaN,NaN,47|52,NaN,0
2169,50.0,AM<sub>2</sub> receptor,Mouse,Q9R1W5|Q9WUP1,47|53,Calcrl|Ramp3,2
2170,50.0,AM<sub>2</sub> receptor,Rat,Q63118|Q9JJ73,47|53,Calcrl|Ramp3,2
2171,50.0,AM<sub>2</sub> receptor,Human,Q16602|O60896,47|53,CALCRL|RAMP3,2


## Acceptance checklist for a candidate complex CURIE

Before adding a mapping to production, record and test:

1. GtoPdb Target ID, target name, species, source component fields, and source
   release;
2. external CURIE, external database/release, taxon, component membership, and
   evidence URL or accession;
3. mapping relation: exact equivalence, cross-reference, or unresolved;
4. cardinality: one-to-one, one-to-many, many-to-one, or no match;
5. intended graph projection: GtoPdb node category and `has_part` versus
   `has_member` relationship;
6. assertion policy: interaction remains target-level unless direct
   component-level evidence exists; and
7. source-equivalence and normalization effects after implementation.

A mapping that cannot pass these checks can remain a research lead, but should
not become an automatic identifier substitution in the ingest.


## Candidate-mapping curation template

This export is a **worklist**, not an accepted mapping. Fill `external_curie`
and the evidence/provenance fields only after review; leave the target row
unmapped when no exact or defensible cross-reference is available.


In [6]:
candidate_path = DATA_DIR / 'gtopdb_complex_mapping_candidates.csv'
candidate_template = target_inventory.assign(
    external_curie=pd.NA,
    mapping_relation=pd.NA,
    evidence=pd.NA,
    confidence=pd.NA,
    review_status='unreviewed',
)
candidate_template.to_csv(candidate_path, index=False)
print(f'Wrote candidate mapping worklist: {candidate_path}')
candidate_template.head(10)


Wrote candidate mapping worklist: /tmp/gtopdb-analysis/gtopdb_complex_mapping_candidates.csv


,Target ID,Target,Target Species,Target UniProt ID,Target Subunit IDs,Target Gene Symbol,component_count,external_curie,mapping_relation,evidence,confidence,review_status
921,378.0,5-HT<sub>3</sub>AB,Human,P46098|O95264,373|374,HTR3A|HTR3B,2,<NA>,<NA>,<NA>,<NA>,unreviewed
930,378.0,5-HT<sub>3</sub>AB,Mouse,P23979|Q9JHJ5,373|374,Htr3a|Htr3b,2,<NA>,<NA>,<NA>,<NA>,unreviewed
1475,2588.0,Activin receptors,Human,P27037|P36896|Q13705|Q8NER5,1787|1790|1791|1792,ACVR1B|ACVR1C|ACVR2A|ACVR2B,4,<NA>,<NA>,<NA>,<NA>,unreviewed
2153,49.0,AM<sub>1</sub> receptor,Human,Q16602|O60895,47|52,CALCRL|RAMP2,2,<NA>,<NA>,<NA>,<NA>,unreviewed
2154,49.0,AM<sub>1</sub> receptor,Rat,Q63118|Q9JHJ1,47|52,Calcrl|Ramp2,2,<NA>,<NA>,<NA>,<NA>,unreviewed
2155,49.0,AM<sub>1</sub> receptor,Mouse,Q9R1W5|Q9WUP0,47|52,Calcrl|Ramp2,2,<NA>,<NA>,<NA>,<NA>,unreviewed
2168,49.0,AM<sub>1</sub> receptor,NaN,NaN,47|52,NaN,0,<NA>,<NA>,<NA>,<NA>,unreviewed
2169,50.0,AM<sub>2</sub> receptor,Mouse,Q9R1W5|Q9WUP1,47|53,Calcrl|Ramp3,2,<NA>,<NA>,<NA>,<NA>,unreviewed
2170,50.0,AM<sub>2</sub> receptor,Rat,Q63118|Q9JJ73,47|53,Calcrl|Ramp3,2,<NA>,<NA>,<NA>,<NA>,unreviewed
2171,50.0,AM<sub>2</sub> receptor,Human,Q16602|O60896,47|53,CALCRL|RAMP3,2,<NA>,<NA>,<NA>,<NA>,unreviewed
